<a href="https://colab.research.google.com/github/RubaEgbaria/RAG/blob/master/video_text_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from gettext import install
import pip

%pip install yt-dlp youtube-transcript-api openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.0 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 32.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 13.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.9/71.9 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 59.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 78.4 MB/s eta 0:00:00:00:0100:01
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=1ae9d

In [ ]:
import json
import re
from datetime import datetime
from youtube_transcript_api import YouTubeTranscriptApi
import yt_dlp
import whisper

def get_video_metadata(url):
    """Extract video metadata using yt-dlp (no API key needed)."""
    ydl_opts = {'quiet': True}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)
        return {
            "id": info['id'],
            "ref": info['webpage_url'],
            "ref_type": "video",
            "title": info['title'],
            "published_date": info.get('upload_date')  # Format: YYYYMMDD
        }

def get_transcript_with_fallback(video_id, url):
    """Try to get transcript via YouTube API; if fails, fallback to Whisper."""
    try:
        # fetch existing subtitles if exists
        transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
        # Prefer manually created subtitles, then auto-generated
        transcript = transcript_list.find_manually_created_transcript(['ar'])
        if not transcript:
            transcript = transcript_list.find_generated_transcript(['ar'])
        lines = [entry['text'] for entry in transcript.fetch()]
        return " ".join(lines), "subtitles"
    except Exception:
        # No subtitles available → use Whisper
        print("No subtitles found. Downloading audio and transcribing with Whisper...")
        ydl_opts = {
            'format': 'bestaudio/best',
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
            }],
            'outtmpl': 'temp_audio.%(ext)s',
            'quiet': True,
        }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])

        model = whisper.load_model("base")
        result = model.transcribe("temp_audio.mp3")
        return result["text"], "whisper"

def save_to_json(data, filename="transcript_data.json"):
    """Save the final dictionary as a JSON file."""
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Saved to {filename}")

def main(youtube_url):
    # 1. Get metadata
    metadata = get_video_metadata(youtube_url)
    video_id = metadata["id"]

    # 2. Get transcript
    transcript_text, source = get_transcript_with_fallback(video_id, youtube_url)

    # 3. Compute word count
    word_count = len(transcript_text.split())

    # 4. Build final JSON object
    output = {
        **metadata,
        "text": transcript_text,
        "word_count": word_count,
        "published_date": datetime.strptime(metadata["published_date"], "%Y%m%d").date().isoformat() if metadata["published_date"] else None,
        "scraped_at": datetime.utcnow().isoformat() + "Z",
        "transcript_source": source  # optional, to know where the text came from
    }

    # 5. Save to JSON
    save_to_json(output)

    # Print summary
    print(f"✅ Done! Title: {output['title']}")
    print(f"   Word count: {word_count}")
    print(f"   Transcript source: {source}")

if __name__ == "__main__":
    # Example usage
    video_url = "https://www.youtube.com/watch?v=qWSnszG36bc"
    main(video_url)

No subtitles found. Downloading audio and transcribing with Whisper...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 377MiB/s]


Saved to transcript_data.json
✅ Done! Title: قريه معاويه وجبال الروحه بين الماضي والحاضر - الاستاذ محمد ابراهيم يوسف ابو علوه اغباريه
   Word count: 3704
   Transcript source: whisper


/tmp/ipykernel_2055807/3482950774.py:74: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "scraped_at": datetime.utcnow().isoformat() + "Z",


In [ ]:
import json
from datetime import datetime
from youtube_transcript_api import YouTubeTranscriptApi
import yt_dlp
import whisper
import os

def get_video_metadata(url):
    ydl_opts = {'quiet': True}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)
        return {
            "id": info['id'],
            "ref": info['webpage_url'],
            "ref_type": "video",
            "title": info['title'],
            "published_date": info.get('upload_date')
        }

def get_transcript_with_fallback(video_id, url):
    # get Arabic subtitles from YouTube
    try:
        transcript_list = YouTubeTranscriptApi.list_transcripts(video_id)
        # Try manual Arabic, then auto-generated Arabic
        transcript = None
        try:
            transcript = transcript_list.find_manually_created_transcript(['ar'])
        except:
            transcript = transcript_list.find_generated_transcript(['ar'])
        if transcript:
            lines = [entry['text'] for entry in transcript.fetch()]
            return " ".join(lines), "youtube_ar"
    except Exception as e:
        print(f"No YouTube Arabic subtitles: {e}")

    # Fallback to Whisper with Arabic forced
    print("Downloading audio and transcribing with Whisper (Arabic)...")
    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'mp3'}],
        'outtmpl': 'temp_audio.%(ext)s',
        'quiet': True,
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

    # Load Whisper model (use 'small' or 'medium' for better Arabic accuracy)
    model = whisper.load_model("large")
    result = model.transcribe("temp_audio.mp3", language="ar", task="transcribe")
    os.remove("temp_audio.mp3")
    return result["text"], "whisper_ar"

def save_to_json(data, filename="transcript_ar.json"):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Saved to {filename}")

def main(youtube_url):
    metadata = get_video_metadata(youtube_url)
    video_id = metadata["id"]
    transcript_text, source = get_transcript_with_fallback(video_id, youtube_url)
    word_count = len(transcript_text.split())  # Note: for Arabic, word splitting is approximate

    output = {
        **metadata,
        "text": transcript_text,
        "word_count": word_count,
        "published_date": datetime.strptime(metadata["published_date"], "%Y%m%d").date().isoformat() if metadata["published_date"] else None,
        "scraped_at": datetime.utcnow().isoformat() + "Z",
        "transcript_source": source
    }
    save_to_json(output)
    print(f"✅ Done! Title: {output['title']}")
    print(f"   Word count: {word_count}")
    print(f"   Transcript source: {source}")

if __name__ == "__main__":
    video_url = input("Enter YouTube URL (Arabic video): ").strip()
    main(video_url)

Enter YouTube URL (Arabic video): https://youtu.be/Tk_sqMBOpY4


No YouTube Arabic subtitles: type object 'YouTubeTranscriptApi' has no attribute 'list_transcripts'


Saved to transcript_ar.json                                
✅ Done! Title: فريده جبارين - زلفه بين الماضي والحاضر
   Word count: 3382
   Transcript source: whisper_ar


/tmp/ipykernel_2055807/3275096017.py:69: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "scraped_at": datetime.utcnow().isoformat() + "Z",


In [ ]:
pip install faster-whisper yt-dlp ffmpeg-python librosa noisereduce

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.7/260.7 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 83.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 80.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 427.6/427.6 kB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 126.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.4/240.4 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.3/491.3 kB 67.4 MB/s eta 0:00:00
   